In [ ]:
%pip install numpy matplotlib scipy > /dev/null
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

import kalman
kalman.set_debug(0)


# Calorimeter Shower Modelling with Kalman Filters

Generic, standalone calorimeter framework — no ROOT required.

Define a detector via a **layer stack of named materials**, simulate
**EM (gamma)** or **hadronic (neutral)** showers with a Monte Carlo sampler,
then reconstruct the shower trajectory and energy with a **5-state Kalman filter**.

**Visualization:**
- Side view (z–y): geometry with active pads shown in **red**
- Hit projections: z–x, z–y scatter + x–y pad energy map
- Resolution curves

---


## Section 1 — Material Library

Built-in database: radiation length (X0), nuclear interaction length (lambda), and
sampling resolution (sigma_s) for each material.
Active scintillator pads (`scint`, `scint_tile`) are shown in **red** in all views.


In [ ]:
from kalman.apps.calorimeter import (
    MATERIALS, MATERIAL_COLORS,
    CaloLayer, CalorimeterGeometry, CalorimeterSampler,
    CalorimeterElectromagneticModel, CalorimeterHadronicModel,
    build_measurements, em_shower_profile, hadronic_shower_profile,
    moliere_radius_mm, ECAL_PRESETS, HCAL_PRESETS,
)

header = f"{'Key':<12} {'Label':<16} {'X0 [mm]':>9} {'lam [mm]':>10} {'sigma_s':>9} {'R_M [mm]':>10}"
print(header)
print("-" * len(header))
for key, mat in MATERIALS.items():
    s  = mat["sigma_s"]
    rm = moliere_radius_mm(key)
    print(f"{key:<12} {mat['label']:<16} {mat['X0_mm']:>9.1f} "
          f"{mat['lambda_int_mm']:>10.1f} {str(round(s,3)) if s else '—':>9} {rm:>10.1f}")


## Section 2 — ZDC-Style HCAL Geometry (Fe + SiPM-on-Tile)

Inspired by the ePIC ZDC design (arXiv:2406.12877):
- **Absorber**: Iron (20 mm per layer)
- **Active**: SiPM-on-tile scintillator pads (3 mm, ~53 mm cell size, ~25 cm²/tile)
- 40 absorber + 40 readout layers → ~5 lambda_int depth

Active pads are shown in **red** in the side view.


In [ ]:
# ZDC HCAL: Fe absorber + SiPM-on-tile scintillator pads
hcal_layers = []
for _ in range(40):
    hcal_layers.append(CaloLayer("fe",          20., is_active=False))
    hcal_layers.append(CaloLayer("scint_tile",   3., is_active=True,
                                  cell_size_mm=53., sigma_sampling=0.50))

hcal = CalorimeterGeometry(hcal_layers, z_start=0., aperture_mm=600.)
print(hcal.summary())
print(f"\nPad position resolution (sigma): {hcal._active_layers[0].pos_resolution_mm:.1f} mm per layer")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 3.8),
                          gridspec_kw={"width_ratios": [4, 1]})
hcal.plot_z(axes[0], show_depth=True)
hcal.plot_xy(axes[1])
fig.suptitle("ZDC HCAL — Fe / SiPM-on-Tile (red = active scintillator pads)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## Section 3 — ECAL Geometry

Options:
- **Crystal ECAL**: 25 x PbWO4 crystals (23 mm each, ~2.6 X0 per crystal)
- **Pb/Scint sampling ECAL**: 25 pairs of 2 mm Pb + 3 mm scintillator pads


In [ ]:
# Crystal ECAL
ecal_crystal = CalorimeterGeometry(
    [CaloLayer("pwo", 23., is_active=True, cell_size_mm=22.)] * 25,
    z_start=700., aperture_mm=200.,
)

# Pb/Scint sampling ECAL (pads in red)
ecal_sampling_layers = []
for _ in range(25):
    ecal_sampling_layers.append(CaloLayer("pb",    2., is_active=False))
    ecal_sampling_layers.append(CaloLayer("scint", 3., is_active=True,
                                           cell_size_mm=10., sigma_sampling=0.20))
ecal_sampling = CalorimeterGeometry(ecal_sampling_layers, z_start=700., aperture_mm=200.)

ecal = ecal_crystal   # switch here
print(ecal.summary())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5),
                          gridspec_kw={"width_ratios": [3, 1]})
ecal.plot_z(axes[0], show_depth=True)
ecal.plot_xy(axes[1])
fig.suptitle("ECAL — PbWO4 Crystal", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## Section 4 — Combined ECAL + HCAL Side View

In [ ]:
combined = CalorimeterGeometry(
    list(hcal._layers) + list(ecal._layers),
    z_start=0., aperture_mm=600.,
)
fig, ax = plt.subplots(figsize=(16, 4))
combined.plot_z(ax, show_depth=True, height_frac=0.42)
ax.set_title("ZDC — HCAL (Fe/SiPM-Tile) + ECAL (PbWO4)  |  red = active pads",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


## Section 5 — EM Shower (gamma) — Simulation + Hit Projections

`CalorimeterSampler.sample()` generates cell-level hits using the Longo-Sestili
shower profile. Hits are distributed across individual pads (snapped to cell grid).


In [ ]:
E_gamma = 10.0  # GeV

hits_em = CalorimeterSampler(ecal).sample(E_gamma, "gamma", seed=42)
print(f"Generated {len(hits_em)} hits for {E_gamma} GeV gamma")
print(f"Total deposited energy: {sum(h['E'] for h in hits_em):.3f} GeV")


In [ ]:
# --- Kalman reconstruction first (for track overlay) ---
z_em, R_em, meas_em = build_measurements(hits_em)

em_model = CalorimeterElectromagneticModel.from_geometry(0, hits_em, ecal)
em_model.engine.forward(meas_em)
em_model.engine.backward()
res_em = em_model.extract_smoothed()

print("EM reconstruction:")
print(f"  E = {res_em['E']:.3f} GeV  (true {E_gamma:.1f})")
print(f"  x0 = {res_em['x0']:+.2f} mm,  y0 = {res_em['y0']:+.2f} mm")
print(f"  chi2/dof = {res_em['chi2_dof']:.2f}")


In [ ]:
# --- Three-panel hit projections with KF track overlay ---
fig, axes = ecal.plot_hits_projections(
    hits_em,
    particle=f"{E_gamma} GeV gamma",
    smoothed_states=em_model.engine.smooth,
)
plt.show()


In [ ]:
# --- Side view with hits overlaid ---
fig, ax = plt.subplots(figsize=(13, 3.5))
ecal.plot_z(ax, show_depth=False, hits=hits_em, particle=f"{E_gamma} GeV gamma")
plt.tight_layout()
plt.show()


## Section 6 — Hadronic Shower (neutral) — Simulation + Hit Projections

Neutrons/neutrals shower over many interaction lengths (lambda_int ~ 186 mm eff.),
producing a much wider, longer shower than EM. The **red pads** in the HCAL lit up
are proportional to deposited energy.


In [ ]:
E_neutral = 10.0  # GeV

hits_had = CalorimeterSampler(hcal).sample(E_neutral, "neutral", seed=7)
print(f"Generated {len(hits_had)} hits for {E_neutral} GeV neutral")
print(f"Total deposited energy: {sum(h['E'] for h in hits_had):.3f} GeV")
print(f"Hadronic transverse spread: {hcal.hadronic_spread_mm:.1f} mm  (cf. lambda_eff/10)")


In [ ]:
# --- Kalman reconstruction ---
z_h, R_h, meas_h = build_measurements(hits_had)

had_model = CalorimeterHadronicModel.from_geometry(0, hits_had, hcal)
had_model.engine.forward(meas_h)
had_model.engine.backward()
res_h = had_model.extract_smoothed()

print("Hadronic reconstruction:")
print(f"  E = {res_h['E']:.3f} GeV  (true {E_neutral:.1f})")
print(f"  x0 = {res_h['x0']:+.2f} mm,  y0 = {res_h['y0']:+.2f} mm")
print(f"  chi2/dof = {res_h['chi2_dof']:.2f}")


In [ ]:
# --- Three-panel projections ---
fig, axes = hcal.plot_hits_projections(
    hits_had,
    particle=f"{E_neutral} GeV neutral",
    smoothed_states=had_model.engine.smooth,
)
plt.show()


In [ ]:
# --- HCAL side view with hits ---
fig, ax = plt.subplots(figsize=(14, 3.8))
hcal.plot_z(ax, show_depth=False, hits=hits_had, particle=f"{E_neutral} GeV neutral")
plt.tight_layout()
plt.show()


## Section 7 — Resolution Study

Theoretical sampling resolution sigma_E/E = sigma_s / sqrt(E) compared to the
ePIC ZDC requirements from the image (50%/sqrt(E) for HCAL, 20%/sqrt(E) for ECAL).


In [ ]:
energies = np.linspace(0.5, 50., 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# HCAL
ax = axes[0]
ax.set_title("HCAL Energy Resolution")
for name, preset in HCAL_PRESETS.items():
    s = preset["sigma_sampling"]
    ax.plot(energies, 100 * s / np.sqrt(energies), lw=2,
            label=f"{name.replace('_','/').upper()}  (sigma_s={s:.2f})")
# ZDC requirement: 50%/sqrt(E) + 5% (spectator neutrons)
ax.plot(energies, 50/np.sqrt(energies) + 5, "k--", lw=1.5,
        label="ZDC req: 50%/√E ⊕ 5%")
# ZDC requirement: 35%/sqrt(E) + 3-5% (pion/kaon)
ax.plot(energies, 35/np.sqrt(energies) + 5, "r--", lw=1.5,
        label="ZDC req: 35%/√E ⊕ 5% (pion/kaon)")
ax.set_xlabel("E [GeV]"); ax.set_ylabel("sigma_E/E [%]")
ax.set_ylim(0, 60); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ECAL
ax2 = axes[1]
ax2.set_title("ECAL Energy Resolution")
for name, preset in ECAL_PRESETS.items():
    s = preset["sigma_sampling"]
    ax2.plot(energies, 100 * s / np.sqrt(energies), lw=2,
             label=f"{name.upper()}  (sigma_s={s:.3f})")
# ZDC requirement: 20%/sqrt(E) + 3% (u-channel)
ax2.plot(energies, 20/np.sqrt(energies) + 3, "k--", lw=1.5,
         label="ZDC req: 20%/√E ⊕ 3%")
ax2.plot(energies, 20/np.sqrt(energies) + 5, "r--", lw=1.5,
         label="ZDC req: 20%/√E ⊕ 5% (pion/kaon)")
ax2.set_xlabel("E [GeV]"); ax2.set_ylabel("sigma_E/E [%]")
ax2.set_ylim(0, 30); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.suptitle("Sampling Resolution vs ZDC Requirements", fontweight="bold")
plt.tight_layout()
plt.show()


## Section 8 — Monte Carlo Resolution Scan

Simulate many events at different energies to measure the reconstructed energy
resolution from the Kalman filter.


In [ ]:
import numpy as np

energies_scan = [2., 5., 10., 20., 50.]
n_trials = 30

results_em  = {}
results_had = {}

for E in energies_scan:
    es_em, es_had = [], []
    for seed in range(n_trials):
        # EM
        h = CalorimeterSampler(ecal).sample(E, "gamma", seed=seed)
        if h:
            _, _, m = build_measurements(h)
            mdl = CalorimeterElectromagneticModel.from_geometry(0, h, ecal)
            mdl.engine.forward(m); mdl.engine.backward()
            es_em.append(mdl.extract_smoothed()["E"])
        # HAD
        h2 = CalorimeterSampler(hcal).sample(E, "neutral", seed=seed)
        if h2:
            _, _, m2 = build_measurements(h2)
            hm = CalorimeterHadronicModel.from_geometry(0, h2, hcal)
            hm.engine.forward(m2); hm.engine.backward()
            es_had.append(hm.extract_smoothed()["E"])
    results_em[E]  = (np.mean(es_em),  np.std(es_em))
    results_had[E] = (np.mean(es_had), np.std(es_had))
    print(f"E={E:4.0f} GeV | EM: {results_em[E][0]:.2f}±{results_em[E][1]:.2f} GeV"
          f"  | HAD: {results_had[E][0]:.2f}±{results_had[E][1]:.2f} GeV")


In [ ]:
E_arr = np.array(energies_scan)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, results, label, color, req_s, req_c in [
    (axes[0], results_em,  "EM (gamma)",   "#1565C0", 0.20, 3.),
    (axes[1], results_had, "HAD (neutral)","#B71C1C", 0.50, 5.),
]:
    means = np.array([results[E][0] for E in energies_scan])
    stds  = np.array([results[E][1] for E in energies_scan])

    ax.errorbar(E_arr, stds / E_arr * 100, fmt="o-", color=color,
                lw=2, ms=7, label=f"KF sigma_E/E ({label})")

    e_fine = np.linspace(0.5, 55., 200)
    ax.plot(e_fine, req_s * 100 / np.sqrt(e_fine) + req_c, "k--", lw=1.5,
            label=f"ZDC req: {int(req_s*100)}%/sqrt(E) + {req_c:.0f}%")

    ax.set_xlabel("True E [GeV]"); ax.set_ylabel("sigma_E/E [%]")
    ax.set_title(label); ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax.set_ylim(0)

plt.suptitle("Kalman Filter Energy Resolution vs ZDC Requirements",
             fontweight="bold")
plt.tight_layout()
plt.show()


## Section 9 — Custom Geometry (W/Si ECAL + Fe/SiPM-Tile HCAL)

Build any configuration. Here: compact W/Si ECAL (EIC-style) upstream
of the Fe/SiPM-tile HCAL.


In [ ]:
wsi_layers = []
for _ in range(30):
    wsi_layers.append(CaloLayer("w",   3.5, is_active=False))
    wsi_layers.append(CaloLayer("si",  0.3, is_active=True,
                                 cell_size_mm=5., sigma_sampling=0.015))

fe_tile_layers = []
for _ in range(40):
    fe_tile_layers.append(CaloLayer("fe",         20., is_active=False))
    fe_tile_layers.append(CaloLayer("scint_tile",  3., is_active=True,
                                     cell_size_mm=53., sigma_sampling=0.50))

custom = CalorimeterGeometry(wsi_layers + fe_tile_layers, z_start=0., aperture_mm=600.)
print(custom.summary())

fig, ax = plt.subplots(figsize=(16, 4))
custom.plot_z(ax, show_depth=True, height_frac=0.40)
ax.set_title("Custom: W/Si ECAL + Fe/SiPM-Tile HCAL  |  red = active pads",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()
